# Task 4 : Model comparison and parameter precision

AICc comparison of 1- vs 2-compartment models. Bootstrap vs MCMC uncertainty on WM MWF.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
from utils import load_preterm_cohort, load_adult_case, mean_decay_curve
from models import (parametric_bootstrap_bi_fixed, mcmc_bi_fixed, akaike_weights)
from analysis import fit_cohort_aicc_1v2
from plotting import plot_aicc_scatter, plot_mcmc_vs_bootstrap

## 4.1 : Load cohort

In [ ]:
preterm, _ = load_preterm_cohort('../data', exclude_file='../data/preterm_exclusions.csv')
print(f"Cohort: {len(preterm)} subjects")

## 4.2 : AICc comparison: 1-comp vs 2-comp

In [ ]:
df_cmp = fit_cohort_aicc_1v2(preterm)

for tissue in ['WM', 'GM', 'CSF']:
    sub = df_cmp[df_cmp['tissue'] == tissue]
    w1, w2 = akaike_weights([sub['AICc_1'].mean(), sub['AICc_2'].mean()])
    print(f"{tissue}: 2-comp preferred in {(sub['dAICc']<0).mean()*100:.0f}% of subjects, "
          f"Akaike weights: 1-comp={w1:.2f}, 2-comp={w2:.2f}")

plot_aicc_scatter(df_cmp)

## 4.3 : Bootstrap vs MCMC precision on WM MWF

In [ ]:
ds = preterm[0]
TE = np.asarray(ds['TE'], float)
wm = mean_decay_curve(ds, 3)

# Bootstrap
v_b, lo_b, hi_b = parametric_bootstrap_bi_fixed(TE, wm, 20.0, 80.0, n_boot=1000)
print(f"Bootstrap: v = {v_b:.3f}, 95% CI [{lo_b:.3f}, {hi_b:.3f}]")

# MCMC
post, v_m, (lo_m, hi_m) = mcmc_bi_fixed(TE, wm, 20.0, 80.0,
    n_iter=50000, burn_in=10000, thin=20, step=0.02)
print(f"MCMC:      v = {v_m:.3f}, 95% CI [{lo_m:.3f}, {hi_m:.3f}]")

plot_mcmc_vs_bootstrap(post, v_m, (lo_m, hi_m), v_b, (lo_b, hi_b), ds)